In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
qs = pd.read_csv('/content/drive/MyDrive/qs-world-rankings-2025.csv')
the = pd.read_csv('/content/drive/MyDrive/The World university rankings 2016-2026.csv')

In [ ]:
qs.head()
qs.info()
qs.describe()
print(qs.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   2025 Rank                       1503 non-null   object 
 1   2024 Rank                       1482 non-null   object 
 2   Institution Name                1503 non-null   object 
 3   Location                        1503 non-null   object 
 4   Location Full                   1503 non-null   object 
 5   Size                            1503 non-null   object 
 6   Academic Reputation             1503 non-null   float64
 7   Employer Reputation             1503 non-null   float64
 8   Faculty Student                 1503 non-null   float64
 9   Citations per Faculty           1503 non-null   float64
 10  International Faculty           1403 non-null   float64
 11  International Students          1445 non-null   float64
 12  International Research Network  15

In [ ]:
the.info()
the.describe()
print(the.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Rank                       200 non-null    int64  
 1   Dense Rank                 200 non-null    object 
 2   University Name            200 non-null    object 
 3   Location                   200 non-null    object 
 4   Overall Score              200 non-null    float64
 5   Teaching                   200 non-null    float64
 6   Research Environment       200 non-null    float64
 7   Research Quality           200 non-null    float64
 8   Industry                   200 non-null    float64
 9   International Outlook      200 non-null    float64
 10  No. of FTE Students        200 non-null    int64  
 11  No. of Students per Staff  200 non-null    float64
 12  International Students     200 non-null    float64
dtypes: float64(8), int64(2), object(3)
memory usage: 2

In [ ]:
qs = qs.drop_duplicates()
the = the.drop_duplicates()

In [ ]:
qs.isnull().sum()
the.isnull().sum()

,0
Rank,0
Dense Rank,0
University Name,0
Location,0
Overall Score,0
Teaching,0
Research Environment,0
Research Quality,0
Industry,0
International Outlook,0


In [ ]:
qs['Institution Name'] = qs['Institution Name'].str.strip().str.lower()
the['University Name'] = the['University Name'].str.strip().str.lower()
qs['Location'] = qs['Location'].str.strip().str.lower()
the['Location'] = the['Location'].str.strip().str.lower()

In [ ]:
merged = pd.merge(qs, the, left_on=['Institution Name', 'Location'], right_on=['University Name', 'Location'], how='outer')

In [ ]:
merged.to_csv('/content/drive/MyDrive/university_cleaned.csv', index=False)

In [ ]:
 df = pd.read_csv('/content/drive/MyDrive/university_cleaned.csv')

In [ ]:
# Clean 'Student Population' (from THE rankings in df) and 'International Students_y' (from THE rankings in df)
def clean_population_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace(',', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    elif '+' in s:
        try:
            return float(s.replace('+', ''))
        except ValueError:
            return None
    elif '~' in s:
        try:
            return float(s.replace('~', ''))
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

def clean_percentage_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace('%', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

df['Student_Population_Cleaned'] = df['No. of FTE Students'].apply(clean_population_range)
df['International_Students_y_Cleaned'] = df['International Students_y'].apply(clean_percentage_range)

df['Global_Rank_Score'] = 100 - df['Rank'] # Using 'Rank' from THE
df['Research_Productivity_Index'] = df['Research Quality'] / df['No. of Students per Staff']
df['Faculty_Student_Ratio'] = df['Faculty Student']
df['Intl_Student_Percentage'] = df['International_Students_y_Cleaned']

In [ ]:
df[['Student_Population_Cleaned','International_Students_y_Cleaned','Global_Rank_Score','Research_Productivity_Index']].describe()

,Student_Population_Cleaned,International_Students_y_Cleaned,Global_Rank_Score,Research_Productivity_Index
count,200.000000,200.000000,200.000000,200.000000
mean,28036.880000,0.252750,-0.500000,6.700124
std,14944.569671,0.138531,57.879185,3.961191
min,674.000000,0.010000,-100.000000,1.463793
25%,17208.500000,0.150000,-50.250000,4.028137
50%,25689.500000,0.225000,-0.500000,5.950213
75%,35362.500000,0.330000,49.250000,7.999359
max,80107.000000,0.720000,99.000000,25.236842


In [ ]:
 from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']] = scaler.fit_transform(
    df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']]
)

In [ ]:
 df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)

In [ ]:
df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)
import plotly.express as px

fig = px.bar(df.sort_values('Performance_Index',ascending=False).head(10),
             x='Institution Name', y='Performance_Index',
             title='Top 10 Universities by Performance Index')
fig.show()

In [ ]:
df.to_csv('/content/drive/MyDrive/university_cleaned.csv')